# Residential Electricity Consumption Forecasting with LSTM

Predicts next-hour electricity consumption for a residential site (Bukit Mahkota, Bangi) from historical electricity readings and hourly weather data.

Pipeline: **data cleaning -> feature engineering -> chronological train/val/test split -> sequence windowing -> LSTM training -> test-set evaluation**.

All heavy lifting lives in the `src/` package (`data_cleaning.py`, `feature_engineering.py`, `dataset.py`, `model.py`, `train.py`, `evaluate.py`) so this notebook stays a thin, readable walk-through — the same functions back the standalone `python -m src.train` / `python -m src.evaluate` scripts.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from src import config, data_cleaning, feature_engineering, dataset, model as model_module, train as train_module, evaluate as evaluate_module

%matplotlib inline
sns.set_theme(style="whitegrid")
train_module.set_seed()
print("TensorFlow:", tf.__version__)

## 2. Load raw data

Two raw sources in `data/`:
- `electricitysupp.csv` — hourly electricity consumption (kWh) for the site
- `weather.xlsx` — hourly weather observations for the same site/period

In [ ]:
elec_raw = pd.read_csv(config.ELECTRICITY_RAW_PATH)
weather_raw = pd.read_excel(config.WEATHER_RAW_PATH)

print("Electricity:", elec_raw.shape)
display(elec_raw.head())
print("\nWeather:", weather_raw.shape)
display(weather_raw.head())

## 3. Data cleaning

`data_cleaning.clean_pipeline()`:
- parses timestamps, sorts, drops duplicate timestamps
- reindexes to a complete hourly range so any gap becomes an explicit row instead of being silently skipped
- normalizes the inconsistent `site_id` naming found in the weather file (single-site dataset)
- drops constant columns (`snow`, `snowdepth` are always 0 in this tropical climate)
- time-interpolates short gaps, clips physically impossible negative values
- inner-joins electricity and weather on `timestamp`

In [ ]:
cleaned = data_cleaning.clean_pipeline()
print(cleaned.shape)
print("Missing values:\n", cleaned.isna().sum().sum())
cleaned.head()

## 4. Exploratory data analysis

In [ ]:
fig, ax = plt.subplots(figsize=(16, 4))
cleaned[config.TARGET_COL].plot(ax=ax)
ax.set_title("Hourly electricity consumption — full period")
ax.set_ylabel(config.TARGET_COL)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

hourly_avg = cleaned.groupby(cleaned.index.hour)[config.TARGET_COL].mean()
hourly_avg.plot(kind="bar", ax=axes[0])
axes[0].set_title("Average consumption by hour of day")
axes[0].set_xlabel("hour")

dow_avg = cleaned.groupby(cleaned.index.dayofweek)[config.TARGET_COL].mean()
dow_avg.index = ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"]
dow_avg.plot(kind="bar", ax=axes[1], color="orange")
axes[1].set_title("Average consumption by day of week")

plt.tight_layout()
plt.show()

In [ ]:
corr_cols = [config.TARGET_COL] + config.WEATHER_FEATURE_COLS
corr = cleaned[corr_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr[[config.TARGET_COL]].sort_values(config.TARGET_COL, ascending=False), annot=True, cmap="coolwarm", ax=ax)
ax.set_title("Correlation of weather variables with electricity consumption")
plt.tight_layout()
plt.show()

## 5. Feature engineering

`feature_engineering.build_feature_set()` adds, all computed strictly from past values (no leakage):
- cyclical time encodings (hour/day-of-week/month as sin/cos, plus `is_weekend`)
- lag features of the target: 1, 2, 3, 24, 168 hours back
- rolling mean/std of the target over short windows

Rows at the start of the series that don't have enough history for the longest lag are dropped.

In [ ]:
featured, feature_cols = feature_engineering.build_feature_set(cleaned)
print(featured.shape, "|", len(feature_cols), "features")
featured[feature_cols + [config.TARGET_COL]].head()

## 6. Chronological train / validation / test split

Split strictly by time (no shuffling across the split boundary) so the model is always evaluated on data that comes *after* what it trained on — the realistic forecasting scenario. Scalers are fit on the training split only and then applied to val/test to avoid leaking their distribution.

In [ ]:
train_df, val_df, test_df = dataset.chronological_split(featured)
print(f"train: {train_df.index.min()} -> {train_df.index.max()} ({len(train_df)} rows)")
print(f"val:   {val_df.index.min()} -> {val_df.index.max()} ({len(val_df)} rows)")
print(f"test:  {test_df.index.min()} -> {test_df.index.max()} ({len(test_df)} rows)")

fig, ax = plt.subplots(figsize=(16, 4))
ax.plot(train_df.index, train_df[config.TARGET_COL], label="train")
ax.plot(val_df.index, val_df[config.TARGET_COL], label="val")
ax.plot(test_df.index, test_df[config.TARGET_COL], label="test")
ax.legend()
ax.set_title("Chronological split")
plt.tight_layout()
plt.show()

## 7. Sequence windowing

Each LSTM input is a **24-hour lookback window** of all features; the target is the electricity reading 1 hour after the window ends. `dataset.prepare_datasets()` fits the scalers on train, scales every split, and builds the sliding windows.

In [ ]:
data = dataset.prepare_datasets(featured, feature_cols)
X_train, y_train = data["X_train"], data["y_train"]
X_val, y_val = data["X_val"], data["y_val"]
X_test, y_test = data["X_test"], data["y_test"]

print("train:", X_train.shape, y_train.shape)
print("val:  ", X_val.shape, y_val.shape)
print("test: ", X_test.shape, y_test.shape)

## 8. Model architecture

In [ ]:
lstm_model = model_module.build_lstm_model(n_timesteps=X_train.shape[1], n_features=X_train.shape[2])
lstm_model.summary()

## 9. Training

Early stopping on validation loss (restoring the best weights), learning-rate reduction on plateau, and checkpointing the best model to `models/lstm_electricity_model.keras`.

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=config.EARLY_STOPPING_PATIENCE, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=config.REDUCE_LR_PATIENCE, min_lr=1e-6),
    tf.keras.callbacks.ModelCheckpoint(filepath=str(config.MODEL_PATH), monitor="val_loss", save_best_only=True),
]

history = lstm_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=config.MAX_EPOCHS,
    batch_size=config.BATCH_SIZE,
    shuffle=True,
    callbacks=callbacks,
    verbose=2,
)

In [ ]:
train_module.plot_training_history(history, config.FIGURES_DIR / "training_history.png")
plt.show()

## 10. Evaluation on the test set

Predictions and targets are inverse-scaled back to real kWh before computing metrics, so MAE/RMSE are directly interpretable in the original units.

In [ ]:
y_pred_scaled = lstm_model.predict(X_test, verbose=0).ravel()

target_scaler = data["target_scaler"]
y_test_real = target_scaler.inverse_transform(y_test.reshape(-1, 1)).ravel()
y_pred_real = target_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()

metrics = evaluate_module.compute_metrics(y_test_real, y_pred_real)
metrics

In [ ]:
test_index = data["index_test"]
evaluate_module.plot_predictions(test_index, y_test_real, y_pred_real, config.FIGURES_DIR / "test_predictions.png")
plt.show()

In [ ]:
evaluate_module.plot_scatter(y_test_real, y_pred_real, config.FIGURES_DIR / "test_scatter.png")
plt.show()

## 11. Conclusion & next steps

- Model, scalers and the feature-column list are saved under `models/` — reload them with `python -m src.evaluate` any time without retraining.
- To retrain end-to-end from the command line: `python -m src.train` then `python -m src.evaluate`.
- Ideas to push accuracy further: multi-step (24h-ahead) forecasting, per-appliance submetering if available, weather forecast inputs instead of observed weather for a true forecasting setup, hyperparameter search over lookback window / LSTM width / dropout.